# Pre-class: Monday Morning — Sarah Opens the Churn Dataset
**⏱ This pre-class notebook takes approximately 15 minutes.**

---

## Scenario: Monday — Marcus's Brief

It's the start of week 3 at NorthStar Retail. Friday afternoon Marcus, the CTO, asked Sarah:

> *"Can you build us a model that predicts churn from our OWN customer data? Something trained on NorthStar behaviour, not on some generic internet corpus?"*

This morning a new file is waiting for her: `data/northstar_churn.csv`. 10,000 customers. 11 features. One target column: `churned` — did they cancel within 30 days?

By Friday she has to:
1. Clean the data so a model can use it.
2. Train her first model.
3. Measure it honestly.

Today is just step zero: **open the file and look around.**

**By the end of this notebook you will be able to:**
- Describe the structure of the NorthStar churn dataset
- Identify which columns have missing values and how many
- Spot the class imbalance
- Have an early gut feel for which features look most predictive

In [ ]:
# Setup — run this cell first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 20)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (11, 4.5)

print("✅ Libraries loaded — you're ready to go!")

## Load the data

The CSV lives in `data/northstar_churn.csv` (one folder above this notebook's data directory). Each row is one customer. The column `churned` is the target — 0 means they stayed, 1 means they cancelled within 30 days.

In [ ]:
df = pd.read_csv("data/northstar_churn.csv")
print(f"Loaded: {len(df):,} rows × {df.shape[1]} columns")
df.head()

### ⏸️ Pause and look

Before scrolling down, scan the table above. Try to answer for yourself:

- Which columns look numerical and which look categorical?
- Do you see any cells with NaN (missing values)?
- Any columns where the values look messy or weird?

*Write your gut-feel observations here (double-click this cell to edit):*

In [ ]:
# What does pandas think each column is?
df.info()

### 💡 What you should notice

- **`region` and `subscription_tier` are `object` dtype** — they're text categories. Models can't compute on text directly; we'll need to encode them as numbers (Tuesday's notebook).
- **`last_login_days_ago` and `avg_review_polarity` are `float64`** — but the Non-Null counts are less than 10,000. Those two columns have missing values.
- **Everything else is numeric and complete.**

Let's see exactly how much is missing.

In [ ]:
missing = df.isna().sum()
missing_pct = (df.isna().mean() * 100).round(1)
missing_table = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_table = missing_table[missing_table["missing_count"] > 0]
print("Columns with missing values:")
print(missing_table.to_string())

### 💡 What this tells us

- About **8% of `last_login_days_ago`** is missing. Plausible interpretation: these are customers who *never logged in* — they made a purchase and never returned to the site. That non-engagement itself is probably predictive of churn.
- About **30% of `avg_review_polarity`** is missing. Plausible interpretation: these are customers who *never wrote a review*. Quieter customers. Whether that predicts churn is an open question.

In Tuesday's notebook, we'll have to decide how to handle these — and we'll also add a "was-missing" flag column so the model can use the missingness itself as a signal.

## The class imbalance

Let's see how often `churned == 1`. This number drives every other decision in the project.

In [ ]:
churn_count = df["churned"].value_counts().rename({0: "stayed", 1: "churned"})
churn_pct   = (df["churned"].value_counts(normalize=True) * 100).round(1).rename({0: "stayed", 1: "churned"})

print("Class breakdown:")
print(pd.concat([churn_count, churn_pct], axis=1, keys=["count", "pct (%)"]).to_string())

fig, ax = plt.subplots(figsize=(7, 4))
churn_count.plot(kind="bar", color=["steelblue", "coral"], edgecolor="white", ax=ax)
ax.set_ylabel("Number of customers")
ax.set_xlabel("")
ax.set_title("Class balance — NorthStar churn dataset")
plt.xticks(rotation=0)
for i, v in enumerate(churn_count):
    ax.text(i, v + 100, f"{v:,}", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

### 💡 What this tells us

- About **12% of customers churned.** That's typical for B2C retail — not absurdly low, but heavily imbalanced.
- **A "model" that always predicts STAYED** would achieve 88% accuracy on this dataset and be totally useless. This is why we cannot trust accuracy alone — and why Thursday's notebook will spend most of its time on confusion-matrix metrics instead.

## A first look at one feature

Let's pick a feature where we expect a story — `tenure_months` (how many months the customer has been with NorthStar). Long-tenure customers should be less likely to churn.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Left: histogram of tenure by churn class
ax = axes[0]
df[df["churned"] == 0]["tenure_months"].hist(bins=30, alpha=0.6, ax=ax,
                                              color="steelblue", label="Stayed")
df[df["churned"] == 1]["tenure_months"].hist(bins=30, alpha=0.7, ax=ax,
                                              color="coral", label="Churned")
ax.set_xlabel("Tenure (months)")
ax.set_ylabel("Number of customers")
ax.set_title("Tenure distribution by churn outcome")
ax.legend()

# Right: churn rate by tenure bucket
ax = axes[1]
df["tenure_bucket"] = pd.cut(df["tenure_months"],
                              bins=[-0.5, 6, 12, 24, 48, 72],
                              labels=["0–6", "7–12", "13–24", "25–48", "49–72"])
churn_by_tenure = df.groupby("tenure_bucket", observed=True)["churned"].mean()
churn_by_tenure.plot(kind="bar", color="indianred", edgecolor="white", ax=ax)
ax.set_ylabel("Churn rate")
ax.set_xlabel("Tenure (months)")
ax.set_title("Churn rate by tenure bucket")
for i, v in enumerate(churn_by_tenure):
    ax.text(i, v + 0.005, f"{v:.1%}", ha="center", fontweight="bold", fontsize=9)
ax.set_ylim(0, churn_by_tenure.max() * 1.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(f"Churn rate by tenure bucket:")
print((churn_by_tenure * 100).round(1).to_string())

### 💡 What this tells us

- **Tenure is a strong predictor.** Customers in their first 6 months churn at roughly 3–4× the rate of those who've been with NorthStar over a year. That gut-feel pattern will show up in the model coefficients on Wednesday.
- **A typical first-time data scientist** would now go on to plot every feature against churn. That's worthwhile but time-consuming. In class, we'll go straight to building the pipeline.

## ✅ Section Summary

| What we did | What it tells us |
|---|---|
| **Looked at the structure** | 10,000 rows × 12 columns. 2 categorical (region, subscription_tier), 9 numeric, 1 target. |
| **Counted missing values** | `last_login_days_ago` ~8% missing · `avg_review_polarity` ~30% missing. Both probably *informative* missingness. |
| **Saw the class balance** | About 12% churners. Accuracy alone will not be a trustworthy metric. |
| **Plotted one feature vs churn** | Tenure is clearly predictive. Early customers churn ~4× more than long-tenured ones. |

**Bring to class:**
1. One column whose values look weird or noisy to you.
2. One column you'd trust as a churn predictor (gut feel).
3. One missing-value column and your guess about why those rows are missing.

---

**In class → Open `02_preprocessing.ipynb` first.** That notebook builds the cleaning pipeline.